# 21 — BoneMarrowWSI-PediatricLeukemia: download the ANNOTATIONS only (no slides)
Candidate real-data source for RQ4 (spatial clustering of blasts in real smears) and RQ3 (certification on real specimens).
Collection on NCI Imaging Data Commons (CC BY 4.0, source DOI 10.5281/zenodo.14933087). The slides are 2.3 TB, but we download only
the annotation series (about 6 MB): monolayer ROIs, consensus cell labels, and unlabeled cell boxes. Plus any small clinical files on Zenodo.
**Runtime:** CPU is fine. Runtime → Run all. When it finishes, download `bmwsi_ann.zip` from Drive to `data/bmwsi/` in the repo.

In [ ]:
!pip -q install idc-index
from google.colab import drive; drive.mount('/content/drive')
import os, json, zipfile, hashlib, requests
OUT = '/content/drive/MyDrive/certified-adaptive-pbs'; os.makedirs(OUT, exist_ok=True)
from idc_index import IDCClient
c = IDCClient(); d = c.index
d = d[(d.collection_id == 'bonemarrowwsi_pediatricleukemia') & (d.Modality == 'ANN')]
keep = d[d.SeriesDescription.isin(['Monolayer regions of interest for cell classification',
                                   'Consensus: cell bounding boxes with cell type labels',
                                   'Unlabeled cell bounding boxes'])]
print(keep.SeriesDescription.value_counts(), round(keep.series_size_MB.sum(), 1), 'MB,', keep.PatientID.nunique(), 'patients')
c.download_from_selection(seriesInstanceUID=list(keep.SeriesInstanceUID), downloadDir='/content/bm_ann')
keep[['PatientID', 'StudyInstanceUID', 'SeriesInstanceUID', 'SeriesDescription', 'series_size_MB']].to_csv('/content/bm_ann/series_index.csv', index=False)
# also record the slide (SM) series per patient: pixel spacing etc. come later only if needed
sm = c.index[(c.index.collection_id == 'bonemarrowwsi_pediatricleukemia') & (c.index.Modality == 'SM')]
sm[['PatientID', 'StudyInstanceUID', 'SeriesInstanceUID', 'series_size_MB']].to_csv('/content/bm_ann/slides_index.csv', index=False)

In [ ]:
# small clinical / metadata files from the Zenodo record (skip anything > 50 MB)
os.makedirs('/content/bm_ann/zenodo', exist_ok=True)
for rid in ['14933087', '17486627']:
    try:
        r = requests.get(f'https://zenodo.org/api/records/{rid}', timeout=60).json()
        for f in r.get('files', []):
            name, size = f['key'], f['size']
            print(rid, name, round(size / 1e6, 1), 'MB')
            if size < 50e6:
                open(f'/content/bm_ann/zenodo/{rid}_{name}', 'wb').write(requests.get(f['links']['self'], timeout=300).content)
    except Exception as e:
        print('zenodo', rid, 'failed:', e)

In [ ]:
zp = f'{OUT}/bmwsi_ann.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk('/content/bm_ann'):
        for f in fs:
            p = os.path.join(root, f); z.write(p, os.path.relpath(p, '/content/bm_ann'))
h = hashlib.sha256(open(zp, 'rb').read()).hexdigest()
print('saved', zp, round(os.path.getsize(zp) / 1e6, 1), 'MB  sha256', h)